In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # notebook lives in feature_engineering/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from feature_engineering.features import (
    FEATURE_COLUMNS, MIN_SPREAD, build_features, latest_snapshot, load_index, load_prices)

prices = load_prices(ROOT / "data" / "prices.csv")
index = load_index(ROOT / "data" / "market_index.csv")
port = pd.read_csv(ROOT / "data" / "portfolio.csv")
features = build_features(prices)

print("Rows:", len(features), "| Tickers:", features["ticker"].nunique(), "| Columns:", features.shape[1])
print("Dates:", features["date"].min().date(), "to", features["date"].max().date())

In [ ]:
num = features.drop(columns=["date", "ticker"])
health = pd.DataFrame({
    "empty_cells": num.isna().sum(),
    "empty_%": (num.isna().mean() * 100).round(1),
    "min": num.min(), "median": num.median(), "max": num.max(),
})
display(health.round(5))

In [ ]:
checks = {
    "no infinite values":                 not np.isinf(num.to_numpy(dtype=float)).any(),
    "one row per ticker and date":        not features.duplicated(["ticker", "date"]).any(),
    "sorted by ticker then date":         features.equals(features.sort_values(["ticker", "date"]).reset_index(drop=True)),
    "adv20 is positive":                  bool((features["adv20"].dropna() > 0).all()),
    "volume_ratio is not negative":       bool((features["volume_ratio"].dropna() >= 0).all()),
    "vol20 is not negative":              bool((features["vol20"].dropna() >= 0).all()),
    "spread20 is at least the 1 bp floor": bool((features["spread20"].dropna() >= MIN_SPREAD).all()),
    "drawdown252 is never above 0":       bool((features["drawdown252"].dropna() <= 1e-12).all()),
    "all 20 portfolio tickers present":   set(port["ticker"]) <= set(features["ticker"]),
    "all expected columns present":       list(features.columns) == FEATURE_COLUMNS,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values()), "At least one check failed, see above"
print("\nAll checks passed")

In [ ]:
cutoff = pd.Timestamp("2022-12-31")
short = build_features(prices[prices["date"] <= cutoff])
full = build_features(prices)
full_cut = full[full["date"] <= cutoff].reset_index(drop=True)
pd.testing.assert_frame_equal(short, full_cut, check_exact=False, rtol=1e-9, atol=1e-12)
print("Identical up to", cutoff.date(), "- no future data leaks into past features")

In [ ]:
today = latest_snapshot(features)
rank = port.merge(today[["ticker", "date", "close", "adv20", "amihud20", "vol20_ann", "spread20"]], on="ticker")
rank["days_at_10pct"] = rank["shares"] / (0.10 * rank["adv20"])
rank["position_cr"] = rank["shares"] * rank["close"] / 1e7
rank = rank.sort_values("days_at_10pct", ascending=False).reset_index(drop=True)
display(rank[["ticker", "sector", "position_cr", "days_at_10pct", "amihud20", "vol20_ann", "spread20"]].round(4))

agreement = rank["days_at_10pct"].rank().corr(rank["amihud20"].rank())
print(f"Rank agreement between days-to-liquidate and Amihud: {agreement:.2f}  (1.00 = identical order)")

px.bar(rank.sort_values("days_at_10pct"), x="days_at_10pct", y="ticker", orientation="h",
       title="Days to liquidate at 10% of daily volume").show()

In [ ]:
before = latest_snapshot(features, "2019-12-31").set_index("ticker")
crash = latest_snapshot(features, "2020-03-24").set_index("ticker")
cmp = pd.DataFrame({
    "adv_change_%": (crash["adv20"] / before["adv20"] - 1) * 100,
    "vol_ann_before": before["vol20_ann"],
    "vol_ann_crash": crash["vol20_ann"],
}).dropna().round(2)
display(cmp.sort_values("vol_ann_crash", ascending=False))
print("Note: in the real crash volume ROSE and volatility exploded. 'Volume halves' is a what-if, not history.")

px.bar(cmp.reset_index().melt(id_vars="ticker", value_vars=["vol_ann_before", "vol_ann_crash"]),
       x="ticker", y="value", color="variable", barmode="group",
       title="Annualised volatility: end of 2019 vs 24 Mar 2020").show()

In [ ]:
pick = ["RELIANCE.NS", "PNB.NS", "IDEA.NS", "TRIDENT.NS"]
sub = features[features["ticker"].isin(pick)]
px.line(sub, x="date", y="adv20", color="ticker", log_y=True, title="20-day average volume (log scale)").show()
px.line(sub, x="date", y="vol20_ann", color="ticker", title="Annualised volatility").show()
px.line(sub, x="date", y="amihud20", color="ticker", log_y=True, title="Amihud illiquidity (log scale)").show()

In [ ]:
floor_share = (features.dropna(subset=["spread20"])
               .assign(at_floor=lambda d: d["spread20"] <= MIN_SPREAD + 1e-12)
               .groupby("ticker")["at_floor"].mean().mul(100).round(0).sort_values())
print("% of days where spread20 sits on the 1 bp floor:")
display(floor_share.to_frame("at_floor_%"))
px.line(features[features["ticker"].isin(["HDFCBANK.NS", "IDEA.NS"])], x="date",
        y=["spread20", "cs_clipped20"], facet_row="ticker", title="spread20 vs cs_clipped20").show()

In [ ]:
cols = ["ret", "hl_range", "volume_ratio", "vol20", "amihud20", "atr_pct", "cs_clipped20",
        "ret_20d", "trend_50", "drawdown252", "volume_ratio_z", "hl_range_z", "vol20_z"]
corr = features[cols].dropna().corr()
px.imshow(corr.round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
          title="Feature correlations").show()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
         .rename("corr").reset_index().sort_values("corr", key=abs, ascending=False))
display(pairs.head(8).round(2))